# ClassifAI Demo

This demo uses a mock occupations dataset to show how ClassifAI matches unlabelled job descriptions to SOC codes using an existing knowledgebase.

In [ ]:
import glob
import os
import string
import pandas as pd
from classifai.indexers import VectorStore
from classifai.vectorisers import FastEmbedVectoriser
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.hooks import (
    HookBase,
    DeduplicationHook,
    CapitalisationStandardisingHook,
)

highlight_kwargs = {"background-color": "#e6ffed"}

### Setup

The cells below load the data and build the vector store.

In [ ]:
# Prepare knowledgebase - VectorStore expects 'label' and 'text' columns
coded_df = pd.read_csv("../data/mock_soc_dataset.csv")
coded_df["label"] = coded_df["soc_code"]
coded_df["text"] = coded_df["role"] + ": " + coded_df["description"]
coded_df.to_csv("../data/mock_vector_store_data.csv", index=False)

# Prepare queries - VectorStore search expects 'id' and 'query' columns
uncoded_df = pd.read_csv("../data/mock_uncoded_soc_responses.csv")
uncoded_df["query"] = uncoded_df["role"] + ": " + uncoded_df["description"]
uncoded_df["id"] = uncoded_df.index
input_data = VectorStoreSearchInput.from_data(uncoded_df)

# We use a local model as mybinder cant download reliably from huggingface or other model repositories.
# Most applications of ClassifAI would just specify a model_name and it will automatically download the files.
search_pattern = "../data/fastembed_cache/models--*--*/snapshots/*"
matching_dirs = glob.glob(search_pattern)

if matching_dirs:
    model_path = matching_dirs[0]
else:
    raise RuntimeError(
        "The pre-cached FastEmbed model directory could not be located. "
        "Please ensure the environment's build step (postBuild) completed successfully "
        "or locate/download the model weights manually. If this issue persists in the "
        "live demo environment, please report it to the ClassifAI team via GitHub."
    )

# Build vector store
vectoriser = FastEmbedVectoriser(
    model_name="BAAI/bge-small-en-v1.5",
    specific_model_path=model_path,
)

soc_vector_store = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    meta_data={"role": str},
    hooks={},
)


---
## 1. Basic Search

Pass a batch of unlabelled job descriptions and get back the best matching SOC code for each one.

In [ ]:
soc_vector_store.search(query=input_data, n_results=1)

Set `n_results` higher to return a ranked shortlist rather than a single match — useful when you want to surface alternatives for review.

In [ ]:
soc_vector_store.search(query=input_data, n_results=3)

---
## 2. Metadata in Results

The `meta_data` parameter controls which extra knowledgebase columns are returned alongside each match. In this example we added the "role" column to the `meta_data` which is why we see it in the search result.

```diff
  soc_vector_store = VectorStore(
      file_name="../data/mock_vector_store_data.csv",
      data_type="csv",
      vectoriser=vectoriser,
+     meta_data={"role": str},
  )
```

In [ ]:
results = soc_vector_store.search(query=input_data, n_results=1)

results.style.set_properties(subset=["role"], **highlight_kwargs)

---
## 3. Hooks

Hooks are pre or post processing functions attached to the vector store - They run automatically on every search call.
We provide some basic ones, but you can make your own!
> This helps keep transformation logic organised and repeatable.

### Capitalisation Standardising Hook

Normalises query casing before embedding. Attach it as to the hook dictionary and it will run on every query automatically.
> The key to the dictionary can be called whatever you want, this allows you to group similar hooks

In [ ]:
soc_vs_lower = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    hooks={
        "search_preprocess": CapitalisationStandardisingHook(
            method="lower", colname="query"
        )
    },
)

# Deliberately inconsistent casing — the hook normalises before embedding
messy_input = VectorStoreSearchInput({
    "id": [0, 1, 2],
    "query": ["TOMATO FARMER", "Machine Learning ENGINEER", "pHd StUdEnT"],
})

soc_vs_lower_result = soc_vs_lower.search(query=messy_input, n_results=1)
soc_vs_lower_result.style.set_properties(subset=["query_text"], **highlight_kwargs)

### Deduplication Hook

With `n_results > 1`, the same doc_label can appear multiple times if several knowledgebase entries share that label. `DeduplicationHook` collapses duplicates to one result per label, keeping only the highest-scoring match.

In [ ]:
# Without deduplication — the same SOC code can appear multiple times per query
soc_result = soc_vector_store.search(query=input_data, n_results=5)

soc_result.style.set_properties(subset=["doc_label"], **highlight_kwargs)

In [ ]:
soc_vs_dedup = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    hooks={"search_postprocess": DeduplicationHook(score_aggregation_method="max")},
)

# Same search — each SOC code now appears at most once per query
soc_dedup_result = soc_vs_dedup.search(query=input_data, n_results=5)[
    ["query_id", "query_text", "doc_label", "score"]
]

soc_dedup_result.style.set_properties(subset=["doc_label"], **highlight_kwargs)

---
## 4. Custom Hooks

Extend `HookBase` to add any preprocessing step you need. The hook receives the search input dataclass and must return the same type.

In [ ]:
class RemovePunctuationHook(HookBase):
    def __call__(self, data):
        data["query"] = [
            q.translate(str.maketrans("", "", string.punctuation))
            for q in data["query"]
        ]
        return data


soc_vs_custom = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    hooks={"search_preprocess": RemovePunctuationHook()},
)

punctuated_input = VectorStoreSearchInput({
    "id": [0, 1],
    "query": ["Tomato... Farmer!!!", "Machine-Learning Engineer (AI/ML)"],
})

soc_custom_result = soc_vs_custom.search(query=punctuated_input, n_results=1)

soc_custom_result.style.set_properties(subset=["query_text"], **highlight_kwargs)